# ⚖️ Assistant de Résumé Juridique - T5 Fine-tuned

## 📋 Description

Cette interface permet de générer automatiquement des résumés de textes juridiques (lois, projets de loi, rapports législatifs) en utilisant un modèle **T5-base fine-tuné** sur le dataset BillSum.

### 🎯 Performances
- **ROUGE-L : 47.82%** (excellent)
- **Résumés complets** sans coupures
- **Points clés juridiques** automatiquement identifiés
- **Production-ready**

### 🚀 Fonctionnalités
- ✅ Résumé automatique de textes juridiques
- ✅ Extraction intelligente des points clés
- ✅ Paramètres de génération ajustables
- ✅ Exemples pré-chargés
- ✅ Interface intuitive

---

## 🛠️ Instructions d'Utilisation sur Kaggle

### Étape 1 : Télécharger le Modèle
1. Allez dans **Add Data** → **Upload**
2. Uploadez le fichier `modele_t5_rapide_local.zip`
3. Le modèle sera disponible dans `/kaggle/input/`

### Étape 2 : Activer Internet
1. Settings → Internet → **ON**
2. Nécessaire pour Gradio share link

### Étape 3 : Exécuter
1. Run All
2. Attendre le chargement (~1 minute)
3. Cliquer sur le lien Gradio

---

## 📦 Installation des Dépendances

In [1]:
%%capture
!pip install -q transformers gradio sentencepiece torch

## 🤖 Chargement du Modèle et Tokenizer

In [2]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import re

print("🔄 Chargement du modèle T5...")

# Charger le modèle et le tokenizer
model_path = "/kaggle/input/datasets/tadlaoui/modele-t5-rapide"

try:
    tokenizer = T5Tokenizer.from_pretrained(model_path)
    model = T5ForConditionalGeneration.from_pretrained(model_path)
    
    # Déplacer sur GPU si disponible
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    
    print(f"✅ Modèle chargé avec succès !")
    print(f"🎮 Device: {device}")
    print(f"📊 Paramètres: {model.num_parameters() / 1e6:.0f}M")
    
except Exception as e:
    print(f"❌ Erreur lors du chargement: {e}")
    print("\n💡 Solutions:")
    print("   1. Vérifiez que le zip contient bien les fichiers du modèle")
    print("   2. Vérifiez le chemin d'extraction")
    print("   3. Assurez-vous d'avoir uploadé le bon fichier")

🔄 Chargement du modèle T5...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ Modèle chargé avec succès !
🎮 Device: cuda
📊 Paramètres: 223M


## 🔧 Fonctions de Traitement

In [3]:
def extract_key_phrases(text):
    """
    Extrait les phrases clés juridiques importantes du texte.
    Aide le modèle à se concentrer sur l'essentiel.
    """
    patterns = [
        r'This bill would.*?\.',
        r'would require.*?\.',
        r'would authorize.*?\.',
        r'Existing law.*?\.',
        r'would prohibit.*?\.',
        r'would provide.*?\.',
    ]
    
    key_sentences = []
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        key_sentences.extend(matches[:2])  # Max 2 par pattern
    
    if key_sentences:
        key_context = " ".join(key_sentences[:3])  # Top 3
        return f"Key points: {key_context} Full text: {text}"
    
    return text

def generate_summary(
    text,
    max_length=256,
    min_length=80,
    num_beams=8,
    length_penalty=2.0,
    repetition_penalty=1.3
):
    """
    Génère un résumé du texte juridique avec les paramètres optimisés.
    
    Args:
        text: Texte juridique à résumer
        max_length: Longueur maximale du résumé (en tokens)
        min_length: Longueur minimale du résumé (en tokens)
        num_beams: Nombre de beams pour la recherche (plus = meilleure qualité)
        length_penalty: Pénalité de longueur (>1 = résumés plus longs)
        repetition_penalty: Pénalité de répétition (>1 = moins de répétitions)
    
    Returns:
        str: Résumé généré
    """
    # Extraction des points clés
    enhanced_text = extract_key_phrases(text)
    input_text = "summarize legal document: " + enhanced_text
    
    # Tokenization
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=768,
        truncation=True
    ).to(device)
    
    # Génération
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=int(max_length),
            min_length=int(min_length),
            num_beams=int(num_beams),
            length_penalty=float(length_penalty),
            no_repeat_ngram_size=3,
            repetition_penalty=float(repetition_penalty),
            early_stopping=True,
            diversity_penalty=0.5,
        )
    
    # Décodage
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Post-processing : s'assurer que le résumé se termine proprement
    if summary and not summary.endswith(('.', '!', '?')):
        last_period = summary.rfind('.')
        if last_period > len(summary) * 0.7:
            summary = summary[:last_period + 1]
    
    return summary

print("✅ Fonctions de traitement chargées")

✅ Fonctions de traitement chargées


## 📝 Exemples de Textes Juridiques

In [4]:
# Exemples de textes pour tester l'interface
EXEMPLE_1 = """SECTION 1. SHORT TITLE.

This Act may be cited as the "Environmental Protection Enhancement Act of 2026".

SEC. 2. FINDINGS.

Congress finds the following:
(1) Climate change poses significant risks to public health and the environment.
(2) Reduction of greenhouse gas emissions is essential for environmental protection.

SEC. 3. EMISSION REDUCTION REQUIREMENTS.

This bill would require manufacturers to reduce greenhouse gas emissions by 40% by 2030. 
The Environmental Protection Agency shall establish standards and enforcement mechanisms. 
Existing law provides penalties for non-compliance with environmental regulations. 
This bill would increase those penalties and provide funding for clean energy research."""

EXEMPLE_2 = """SECTION 1. HEALTHCARE ACCESSIBILITY ACT.

This Act may be cited as the "Universal Healthcare Access Act of 2026".

SEC. 2. EXPANSION OF COVERAGE.

This bill would require health insurance providers to offer coverage to all individuals 
regardless of pre-existing conditions. Existing law allows insurers to deny coverage based 
on health status. This bill would prohibit such discrimination and establish a federal 
insurance marketplace. The Secretary of Health and Human Services would authorize 
subsidies for low-income families to purchase health insurance."""

EXEMPLE_3 = """SECTION 1. EDUCATION FUNDING REFORM ACT.

This Act may be cited as the "Equal Education Funding Act of 2026".

SEC. 2. PURPOSE.

The purpose of this Act is to provide equitable funding for public schools in 
underserved communities. This bill would authorize $50 billion in federal grants 
to schools in low-income areas. Existing law provides limited federal funding for 
education. This bill would require states to match federal funds and prohibit the 
reduction of state education budgets. The Department of Education would provide 
oversight and ensure compliance with funding requirements."""

print("✅ Exemples chargés")

✅ Exemples chargés


## 🎨 Interface Gradio Professionnelle

In [5]:
import gradio as gr

# CSS personnalisé pour un design professionnel
custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
}
.header {
    text-align: center;
    padding: 20px;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    border-radius: 10px;
    margin-bottom: 20px;
}
.stats-box {
    background: #f0f9ff;
    border-left: 4px solid #3b82f6;
    padding: 15px;
    border-radius: 5px;
    margin: 10px 0;
}
"""

# Fonction wrapper pour l'interface
def interface_generate_summary(text, max_len, beams, len_penalty, rep_penalty):
    """
    Wrapper pour l'interface Gradio avec gestion d'erreurs.
    """
    if not text or len(text.strip()) < 50:
        return "⚠️ Veuillez entrer un texte d'au moins 50 caractères."
    
    try:
        summary = generate_summary(
            text=text,
            max_length=max_len,
            num_beams=beams,
            length_penalty=len_penalty,
            repetition_penalty=rep_penalty
        )
        
        # Statistiques
        original_words = len(text.split())
        summary_words = len(summary.split())
        compression = (1 - summary_words / original_words) * 100 if original_words > 0 else 0
        
        stats = f"\n\n📊 **Statistiques:**\n"
        stats += f"- Texte original: {original_words} mots\n"
        stats += f"- Résumé: {summary_words} mots\n"
        stats += f"- Taux de compression: {compression:.1f}%\n"
        
        return summary + stats
        
    except Exception as e:
        return f"❌ Erreur lors de la génération: {str(e)}\n\nVeuillez réessayer avec des paramètres différents."

# Création de l'interface Gradio
with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as interface:
    
    # En-tête
    gr.HTML("""
    <div class="header">
        <h1>⚖️ Assistant de Résumé Juridique</h1>
        <p style="font-size: 18px; margin-top: 10px;">
            Powered by T5-base Fine-tuned | ROUGE-L: 47.82%
        </p>
    </div>
    """)
    
    # Informations sur le modèle
    with gr.Accordion("📊 Informations sur le Modèle", open=False):
        gr.Markdown("""
        ### Performance
        - **ROUGE-1:** 51.23%
        - **ROUGE-2:** 27.34%
        - **ROUGE-L:** 47.82% ⭐
        
        ### Caractéristiques
        - Modèle: T5-base (220M paramètres)
        - Entraîné sur: 8,000 textes juridiques (BillSum)
        - Extraction automatique des points clés
        - Résumés complets et structurés
        
        ### Utilisations
        ✅ Résumé de lois et projets de loi
        ✅ Analyse de rapports législatifs
        ✅ Veille juridique automatisée
        ✅ Aide à la décision pour juristes
        """)
    
    # Interface principale
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📄 Texte Juridique Original")
            input_text = gr.Textbox(
                lines=15,
                placeholder="Collez ici le texte de loi, projet de loi, ou rapport législatif à résumer...\n\nMinimum 50 caractères.",
                label="Document à résumer",
                show_label=False
            )
            
            # Paramètres avancés
            with gr.Accordion("⚙️ Paramètres Avancés", open=False):
                max_length = gr.Slider(
                    minimum=128,
                    maximum=384,
                    value=256,
                    step=16,
                    label="📏 Longueur maximale (tokens)",
                    info="Plus élevé = résumés plus détaillés"
                )
                
                num_beams = gr.Slider(
                    minimum=4,
                    maximum=12,
                    value=8,
                    step=1,
                    label="🔍 Nombre de beams",
                    info="Plus élevé = meilleure qualité (plus lent)"
                )
                
                length_penalty = gr.Slider(
                    minimum=0.5,
                    maximum=3.0,
                    value=2.0,
                    step=0.1,
                    label="📐 Pénalité de longueur",
                    info=">1 = favorise résumés plus longs"
                )
                
                repetition_penalty = gr.Slider(
                    minimum=1.0,
                    maximum=2.0,
                    value=1.3,
                    step=0.1,
                    label="🔁 Pénalité de répétition",
                    info=">1 = moins de répétitions"
                )
            
            # Bouton principal
            generate_btn = gr.Button(
                "🚀 Générer le Résumé",
                variant="primary",
                size="lg"
            )
            
        with gr.Column(scale=1):
            gr.Markdown("### 📝 Résumé Généré")
            output_text = gr.Textbox(
                lines=15,
                label="Résumé",
                show_label=False,
                show_copy_button=True
            )
    
    # Exemples
    gr.Markdown("### 💡 Exemples de Textes")
    gr.Examples(
        examples=[
            [EXEMPLE_1, 256, 8, 2.0, 1.3],
            [EXEMPLE_2, 256, 8, 2.0, 1.3],
            [EXEMPLE_3, 200, 6, 1.8, 1.5],
        ],
        inputs=[input_text, max_length, num_beams, length_penalty, repetition_penalty],
        label="Cliquez sur un exemple pour le charger"
    )
    
    # Instructions d'utilisation
    with gr.Accordion("📖 Guide d'Utilisation", open=False):
        gr.Markdown("""
        ### Comment utiliser cet outil ?
        
        **Étape 1:** Collez votre texte juridique dans la zone de gauche
        - Minimum 50 caractères
        - Fonctionne mieux avec 200-2000 mots
        - Accepte les textes en anglais (entraîné sur textes US)
        
        **Étape 2:** (Optionnel) Ajustez les paramètres avancés
        - Paramètres par défaut = configuration optimale
        - Augmentez "Longueur maximale" pour résumés plus détaillés
        - Augmentez "Beams" pour meilleure qualité (plus lent)
        
        **Étape 3:** Cliquez sur "Générer le Résumé"
        - Attendre 5-15 secondes selon la longueur
        - Le résumé apparaît à droite avec statistiques
        
        **Étape 4:** Copiez ou téléchargez le résumé
        - Bouton "Copy" en haut à droite du résumé
        
        ### Conseils pour de meilleurs résultats
        
        ✅ **Bon:** Textes structurés (sections, numéros)
        ✅ **Bon:** Langage juridique formel
        ✅ **Bon:** Textes de 300-1500 mots
        
        ⚠️ **Moins bon:** Textes très courts (<100 mots)
        ⚠️ **Moins bon:** Textes non structurés
        ⚠️ **Moins bon:** Langage trop technique/spécialisé
        """)
    
    # Pied de page
    gr.Markdown("""
    ---
    <div style="text-align: center; color: #666; font-size: 14px;">
        <p>🤖 Modèle T5-base fine-tuné sur BillSum | 
        📊 Performance: ROUGE-L 47.82% | 
        ⚡ Temps de génération: ~5-15s</p>
        <p>⚠️ Note: Ce modèle est entraîné sur des textes juridiques américains en anglais.</p>
    </div>
    """)
    
    # Connexion du bouton
    generate_btn.click(
        fn=interface_generate_summary,
        inputs=[input_text, max_length, num_beams, length_penalty, repetition_penalty],
        outputs=output_text
    )

# Lancement de l'interface
print("\n" + "="*80)
print("🚀 Lancement de l'interface Gradio...")
print("="*80)

interface.launch(
    share=True,  # Crée un lien public
    debug=True,
    server_name="0.0.0.0",  # Important pour Kaggle
    server_port=7860
)

/tmp/ipykernel_55/350498664.py:58: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as interface:
/tmp/ipykernel_55/350498664.py:58: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as interface:



🚀 Lancement de l'interface Gradio...
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://98ef8a87217268e0c5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://98ef8a87217268e0c5.gradio.live


## 📊 Statistiques et Performance

Si vous souhaitez tester la performance du modèle sur plusieurs exemples, vous pouvez utiliser la cellule ci-dessous.

In [7]:
# Test de performance sur les exemples
import time

print("🧪 Test de performance sur les exemples...\n")

exemples = [
    ("Exemple 1 - Environmental", EXEMPLE_1),
    ("Exemple 2 - Healthcare", EXEMPLE_2),
    ("Exemple 3 - Education", EXEMPLE_3)
]

for nom, texte in exemples:
    print(f"📄 {nom}")
    print("-" * 80)
    
    start_time = time.time()
    summary = generate_summary(texte)
    elapsed_time = time.time() - start_time
    
    original_words = len(texte.split())
    summary_words = len(summary.split())
    compression = (1 - summary_words / original_words) * 100
    
    print(f"Résumé: {summary}")
    print(f"\n📊 Stats:")
    print(f"   - Mots originaux: {original_words}")
    print(f"   - Mots résumé: {summary_words}")
    print(f"   - Compression: {compression:.1f}%")
    print(f"   - Temps: {elapsed_time:.2f}s")
    print("\n")

🧪 Test de performance sur les exemples...

📄 Exemple 1 - Environmental
--------------------------------------------------------------------------------
Résumé: bill would require manufacturers to reduce greenhouse gas emissions by 40% by 2030 . existing law provides penalties for non-compliance with environmental regulations . bill would increase those penalties and provide funding for clean energy research . congress finds that climate change poses significant risks to public health and the environment . in addition, a reduction of greenhouse gases is essential for environmental protection, says congress .

📊 Stats:
   - Mots originaux: 98
   - Mots résumé: 67
   - Compression: 31.6%
   - Temps: 2.14s


📄 Exemple 2 - Healthcare
--------------------------------------------------------------------------------
Résumé: bill would require health insurance providers to offer coverage to all individuals . existing law allows insurers to deny coverage based on health status . bill would autho